# 02 Data Cleaning and Feature Engineering

## Loan Default Prediction and Credit Risk Scorecard

This notebook prepares the cleaned modeling dataset. It removes leakage variables, cleans selected borrower and loan features, engineers credit-risk features, reviews missing values, and saves the final dataset for modeling.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

In [3]:
file_path = "../data/raw/accepted_2007_to_2018Q4.csv"

df = pd.read_csv(file_path, low_memory=False)

df.shape

(2260701, 151)

In [4]:
def map_default_status(status):
    if status in [
        "Charged Off",
        "Default",
        "Late (31-120 days)",
        "Does not meet the credit policy. Status:Charged Off"
    ]:
        return 1
    elif status in [
        "Fully Paid",
        "Does not meet the credit policy. Status:Fully Paid"
    ]:
        return 0
    else:
        return np.nan

df["default_flag"] = df["loan_status"].apply(map_default_status)

df_model = df[df["default_flag"].notna()].copy()
df_model["default_flag"] = df_model["default_flag"].astype(int)

df_model["default_flag"].value_counts(normalize=True).mul(100).round(2)

default_flag
0    78.77
1    21.23
Name: proportion, dtype: float64

In [5]:
leakage_cols = [
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "out_prncp",
    "out_prncp_inv",
    "last_credit_pull_d",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term",
    "debt_settlement_flag",
    "hardship_flag",
    "hardship_type",
    "hardship_reason",
    "hardship_status"
]

existing_leakage_cols = [col for col in leakage_cols if col in df_model.columns]

df_model = df_model.drop(columns=existing_leakage_cols)

len(existing_leakage_cols), existing_leakage_cols

(23,
 ['total_pymnt',
  'total_pymnt_inv',
  'total_rec_prncp',
  'total_rec_int',
  'total_rec_late_fee',
  'recoveries',
  'collection_recovery_fee',
  'last_pymnt_d',
  'last_pymnt_amnt',
  'next_pymnt_d',
  'out_prncp',
  'out_prncp_inv',
  'last_credit_pull_d',
  'settlement_status',
  'settlement_date',
  'settlement_amount',
  'settlement_percentage',
  'settlement_term',
  'debt_settlement_flag',
  'hardship_flag',
  'hardship_type',
  'hardship_reason',
  'hardship_status'])

In [6]:
selected_features = [
    "loan_amnt",
    "funded_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "earliest_cr_line",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "application_type",
    "mort_acc",
    "pub_rec_bankruptcies",
    "issue_d"
]

selected_features = [col for col in selected_features if col in df_model.columns]

model_df = df_model[selected_features + ["default_flag"]].copy()

model_df.shape

(1369566, 26)

In [7]:
if "int_rate" in model_df.columns:
    model_df["int_rate"] = (
        model_df["int_rate"]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace("nan", np.nan)
        .astype(float)
    )

if "revol_util" in model_df.columns:
    model_df["revol_util"] = (
        model_df["revol_util"]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace("nan", np.nan)
        .astype(float)
    )

if "term" in model_df.columns:
    model_df["term_months"] = (
        model_df["term"]
        .astype(str)
        .str.replace(" months", "", regex=False)
        .str.strip()
        .astype(int)
    )

In [8]:
def clean_emp_length(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip()
    
    if value == "< 1 year":
        return 0
    elif value == "10+ years":
        return 10
    elif value == "n/a":
        return np.nan
    else:
        return (
            value
            .replace(" years", "")
            .replace(" year", "")
            .strip()
        )

model_df["emp_length_years"] = model_df["emp_length"].apply(clean_emp_length)
model_df["emp_length_years"] = pd.to_numeric(model_df["emp_length_years"], errors="coerce")

model_df[["emp_length", "emp_length_years"]].head(10)

,emp_length,emp_length_years
0,10+ years,10.0
1,10+ years,10.0
2,10+ years,10.0
4,3 years,3.0
5,4 years,4.0
6,10+ years,10.0
7,10+ years,10.0
8,6 years,6.0
9,10+ years,10.0
12,3 years,3.0


In [9]:
model_df["issue_d"] = pd.to_datetime(model_df["issue_d"], errors="coerce")
model_df["earliest_cr_line"] = pd.to_datetime(model_df["earliest_cr_line"], errors="coerce")

model_df["issue_year"] = model_df["issue_d"].dt.year
model_df["issue_month"] = model_df["issue_d"].dt.month

model_df["credit_history_months"] = (
    (model_df["issue_d"].dt.year - model_df["earliest_cr_line"].dt.year) * 12
    + (model_df["issue_d"].dt.month - model_df["earliest_cr_line"].dt.month)
)

model_df[["issue_d", "earliest_cr_line", "credit_history_months"]].head()

C:\Users\jsjat\AppData\Local\Temp\ipykernel_17488\3389709884.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  model_df["issue_d"] = pd.to_datetime(model_df["issue_d"], errors="coerce")
C:\Users\jsjat\AppData\Local\Temp\ipykernel_17488\3389709884.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  model_df["earliest_cr_line"] = pd.to_datetime(model_df["earliest_cr_line"], errors="coerce")


,issue_d,earliest_cr_line,credit_history_months
0,2015-12-01,2003-08-01,148.0
1,2015-12-01,1999-12-01,192.0
2,2015-12-01,2000-08-01,184.0
4,2015-12-01,1998-06-01,210.0
5,2015-12-01,1987-10-01,338.0


In [10]:
model_df["loan_income_ratio"] = model_df["loan_amnt"] / model_df["annual_inc"]

model_df["installment_income_ratio"] = (
    model_df["installment"] * 12
) / model_df["annual_inc"]

model_df["high_dti_flag"] = np.where(model_df["dti"] >= 30, 1, 0)

model_df["high_revol_util_flag"] = np.where(model_df["revol_util"] >= 80, 1, 0)

model_df["recent_inquiry_flag"] = np.where(model_df["inq_last_6mths"] >= 2, 1, 0)

model_df["long_term_flag"] = np.where(model_df["term_months"] == 60, 1, 0)

grade_mapping = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6,
    "G": 7
}

model_df["grade_risk_numeric"] = model_df["grade"].map(grade_mapping)

In [11]:
model_df["income_band"] = pd.cut(
    model_df["annual_inc"],
    bins=[0, 40000, 80000, 120000, np.inf],
    labels=["Low Income", "Middle Income", "Upper Middle Income", "High Income"]
)

model_df["dti_band"] = pd.cut(
    model_df["dti"],
    bins=[-np.inf, 10, 20, 30, np.inf],
    labels=["Low DTI", "Moderate DTI", "High DTI", "Very High DTI"]
)

model_df["credit_history_band"] = pd.cut(
    model_df["credit_history_months"],
    bins=[-np.inf, 36, 84, 180, np.inf],
    labels=[
        "Short Credit History",
        "Moderate Credit History",
        "Established Credit History",
        "Long Credit History"
    ]
)

In [12]:
model_df = model_df.replace([np.inf, -np.inf], np.nan)

In [13]:
missing_after_cleaning = (
    model_df.isnull()
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .reset_index()
)

missing_after_cleaning.columns = ["column", "missing_pct"]

missing_after_cleaning.head(40)

,column,missing_pct
0,emp_length,5.870546
1,emp_length_years,5.870546
2,mort_acc,3.652982
3,pub_rec_bankruptcies,0.099667
4,revol_util,0.067467
5,dti_band,0.029060
6,dti,0.029060
7,income_band,0.028330
8,installment_income_ratio,0.028330
9,loan_income_ratio,0.028330


In [14]:
missing_after_cleaning.to_csv("../data/outputs/missing_after_cleaning.csv", index=False)

In [15]:
numeric_cols = model_df.select_dtypes(include=["int64", "float64"]).columns.tolist()

model_df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

,count,mean,std,min,1%,5%,50%,95%,99%,max
loan_amnt,1369566.0,14448.774283,8737.506570,500.000000,1500.000000,3200.000000,12000.000000,33000.000000,35000.000000,40000.00
funded_amnt,1369566.0,14440.100495,8733.728330,500.000000,1500.000000,3200.000000,12000.000000,33000.000000,35000.000000,40000.00
int_rate,1369566.0,13.279112,4.784094,5.310000,5.320000,6.540000,12.790000,22.200000,26.490000,30.99
installment,1369566.0,438.759304,262.010546,4.930000,52.873000,108.810000,375.430000,964.665000,1223.430000,1719.83
annual_inc,1369562.0,76250.092671,70310.781229,0.000000,18000.000000,28000.000000,65000.000000,155000.000000,252000.000000,10999200.00
dti,1369168.0,18.303684,11.311968,-1.000000,1.760000,4.960000,17.620000,33.020000,38.560000,999.00
delinq_2yrs,1369537.0,0.318285,0.879336,0.000000,0.000000,0.000000,0.000000,2.000000,4.000000,39.00
inq_last_6mths,1369536.0,0.661674,0.958930,0.000000,0.000000,0.000000,0.000000,3.000000,4.000000,33.00
open_acc,1369537.0,11.591343,5.482311,0.000000,3.000000,5.000000,11.000000,22.000000,29.000000,90.00
pub_rec,1369537.0,0.215452,0.604005,0.000000,0.000000,0.000000,0.000000,1.000000,2.000000,86.00


In [16]:
cap_cols = [
    "annual_inc",
    "dti",
    "revol_util",
    "loan_income_ratio",
    "installment_income_ratio",
    "credit_history_months"
]

for col in cap_cols:
    if col in model_df.columns:
        lower = model_df[col].quantile(0.01)
        upper = model_df[col].quantile(0.99)
        model_df[col] = model_df[col].clip(lower=lower, upper=upper)

In [17]:
final_features = [
    "loan_amnt",
    "funded_amnt",
    "term_months",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "grade_risk_numeric",
    "emp_length_years",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "application_type",
    "mort_acc",
    "pub_rec_bankruptcies",
    "issue_year",
    "issue_month",
    "credit_history_months",
    "loan_income_ratio",
    "installment_income_ratio",
    "high_dti_flag",
    "high_revol_util_flag",
    "recent_inquiry_flag",
    "long_term_flag",
    "income_band",
    "dti_band",
    "credit_history_band"
]

final_features = [col for col in final_features if col in model_df.columns]

final_model_df = model_df[final_features + ["default_flag"]].copy()

final_model_df.shape

(1369566, 37)

In [18]:
final_model_df.to_csv("../data/processed/loan_modeling_dataset.csv", index=False)

print("Clean modeling dataset saved.")
print("Shape:", final_model_df.shape)

Clean modeling dataset saved.
Shape: (1369566, 37)


In [19]:
feature_list = pd.DataFrame({
    "feature": final_features,
    "included_in_modeling_dataset": "Yes"
})

feature_list.to_csv("../data/outputs/final_feature_list.csv", index=False)

feature_list

,feature,included_in_modeling_dataset
0,loan_amnt,Yes
1,funded_amnt,Yes
2,term_months,Yes
3,int_rate,Yes
4,installment,Yes
5,grade,Yes
6,sub_grade,Yes
7,grade_risk_numeric,Yes
8,emp_length_years,Yes
9,home_ownership,Yes


# Data Cleaning and Feature Engineering Summary

## Key Actions Completed

- Created binary default target using final loan outcomes.
- Included fully paid and charged-off loans that did not meet credit policy because their outcomes are known.
- Removed post-origination repayment and hardship variables to prevent data leakage.
- Cleaned interest rate, revolving utilization, loan term, employment length, and date fields.
- Engineered credit-risk features including loan-to-income ratio, installment-to-income ratio, credit history length, DTI flags, utilization flags, inquiry flags, and long-term loan flags.
- Created business-friendly bands for income, DTI, and credit history.
- Capped extreme numeric values at the 1st and 99th percentiles.
- Saved the final modeling dataset for EDA and machine learning.

## Next Step

The next notebook will perform exploratory data analysis to understand default patterns across borrower, loan, and credit profile segments.